# Поведенческий скоринг для предсказания дефолта по кредиту

Проект производит поведенческий скоринг пользователей банка, определяя вероятность дефолта, то есть задержки платежей по кредиту сверх 90 дней.

## Содержание
1. <a href="#problem_statement">✅ Постановка задачи</a>
1. <a href="#environment">✅ Подготовка окружения</a>
1. <a href="#explore_data">Исследовательский анализ данных</a>
1. <a href="#dataset">Формирование датасета</a>
1. <a href="#modelling">Моделирование</a>
1. <a href="#calibration">Калибровка модели</a>
1. <a href="#find_threshold">Поиск порога решения</a>
1. <a href="#conf_matrix">Анализ матрицы ошибок</a>
1. <a href="#final_model">Фиксирование итоговой модели</a>
1. <a href="#feature_importance">Анализ важности признаков</a>
1. <a href="#final_report">Выводы по проекту</a>











<a id="problem_statement"></a>

## Постановка задачи машинного обучения

* Напишите, что конкретно будете делать, пользуясь терминами машинного обучения:
  * какие модели будете использовать;
  * какие методы работы с данными применять;
  * как планируете решать поставленную задачу.


### Бизнес-контекст

Банк хочет заранее понимать, какие действующие клиенты могут уйти в серьезную просрочку по кредиту. Обычный кредитный скоринг помогает только на этапе выдачи кредита, а здесь задача другая - следить за уже существующим портфелем и вовремя замечать рост риска.

Если модель будет хорошо находить таких клиентов заранее, банк сможет лучше управлять резервами, снизить риски и не замораживать лишние деньги.

### Цель исследования

Нужно построить модель, которая по данным о клиенте в конкретный месяц будет предсказывать, случится ли у него в ближайшие 12 месяцев просрочка 90+ дней.

### Постановка задачи машинного обучения

Это задача бинарной классификации:

- 1 - если в следующие 12 месяцев у клиента появится просрочка 90+ дней;
- 0 - если нет.

Каждая строка - это клиент в определенный месяц, поэтому здесь важно учитывать временную структуру данных и аккуратно собирать признаки, чтобы не было утечки данных из будущего.

В данные входят разные группы признаков:

- информация о клиенте;
- параметры кредита;
- траты по MCC-категориям;
- кредитный рейтинг;
- наличие ипотеки;
- макроэкономические показатели;
- новые агрегированные признаки, которые будут созданы в ходе проекта.

Сначала все таблицы нужно объединить в одну итоговую выборку, потом подготовить признаки и целевую переменную.

### Какие модели будем сравнивать

Планируется сравнить логистическую регрессию как базовую модель и Random Forest как основную модель для итогового решения.

Модели будут обучаться:
- на исходных данных;
- на данных с балансировкой классов.

Для итогового решения будет использоваться Random Forest с подбором гиперпараметров через Optuna.

Так как данные временные, для кросс-валидации нужно использовать GroupTimeSeriesSplit, чтобы не перемешивать прошлое и будущее.

### Метрики

Главные метрики здесь бизнесовые:

- Approval rate - доля клиентов, которых модель считает надежными;
- Default rate - доля дефолтов среди тех, кого модель посчитала надежными;
- Missed defaults rate - доля дефолтов, которые модель пропустила.

Дополнительно можно смотреть ROC-AUC или accuracy для общего сравнения моделей.

### Что нужно получить в итоге

Нужно подобрать такую модель и такой порог вероятности, чтобы выполнялись условия:

- Approval rate не меньше 65%;
- Default rate не больше 2%;
- Missed defaults rate не больше 4%.

После этого модель нужно проверить на тестовой выборке и посмотреть, насколько стабильно она работает.



### Подготовка рабочего пространства

* Дайте этой тетрадке Jupyter название, которое корректно отражает суть проекта.
* Опишите постановку задачи своими словами, коротко зафиксировав основной смысл.
* Опишите данные, которые даны для решения задачи.

<a id="environment"></a>

## Подготовка окружения и загрузка данных

### Загрузка необходимых библиотек

* Загрузите все библиотеки, необходимые для выполнения проекта.

In [1]:
! rm requirements.txt

In [2]:
from pathlib import Path

requirements_file = Path('requirements.txt')
requirements = [
    'phik==0.12.5',
    'joblib==1.5.3',
    'scikit-learn==1.6.1',
    'seaborn==0.13.2',
    'optuna==4.8.0',
    'plotly==6.7.0',
    'humanfriendly==10.0',
]
if not requirements_file.exists():
    with open(requirements_file, 'w') as f:
        f.write('\n'.join(requirements))
        print(f'{requirements_file} created')
else:
    print(f'{requirements_file} exists')

print('Установка зависимостей...')
!pip install -r requirements.txt
print('Зависимости успешно установлены!')


requirements.txt created
Установка зависимостей...

[notice] A new release of pip is available: 25.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Зависимости успешно установлены!


In [3]:
import os
import requests
from datetime import datetime
import optuna
import plotly
import joblib
from phik import phik_matrix

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import FunctionTransformer

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate, cross_val_score, KFold
from sklearn.metrics import (
    root_mean_squared_error,
    mean_squared_error,
    r2_score,
    mean_absolute_error,
    make_scorer,
)

from humanfriendly import format_size


/Users/ngsmirnov/nikki/projects/practicum/sprint15_randforest_credit_default_classif/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Подготовка параметров тетрадки и утилит:

In [4]:
TARGET_COL_NAME = 'Rented Bike Count'
RANDOM_STATE = 153
np.random.seed(RANDOM_STATE)

pd.set_option('display.max_columns', None) # выводить все колонки
pd.set_option('display.max_colwidth', 500) # выводить больше символов в ячейке

#### Заргузка ресурсов
class ResourceLoader:
    """
    Класс для безопасной загрузки ресурсов по http: датасет и произвольный файл.
    Если файл уже был скачан ранее и сохранен в локальной файловой системе, то
    загрузка из удаленного источника не производится.

    Для загрузки датасета: load_dataset(dataset_url, local_file, local_path='datasets'): dataframe
    Для загрузки файла: load_resource(resource_url, local_path, local_file): path
    """

    def __init__(self):
        pass

    def load_resource(self, resource_url, local_path, local_file):
        local_resource_file = f'{local_path}/{local_file}'
        if os.path.exists(local_resource_file):
            print(f'Файл {local_resource_file} уже существует')
            return local_resource_file

        os.makedirs(local_path, exist_ok=True)
        print(f'Файл не найден. Загружаем файл из {resource_url}')
        response = requests.get(resource_url)
        if response.status_code == 200:
            with open(local_resource_file, 'wb') as f:
                f.write(response.content)
            file_size = format_size(os.path.getsize(local_resource_file))
            print(f'Файл успешно загружен в {local_resource_file},',
                  f'размер файла: {file_size}')
            return local_resource_file
        else:
            raise NetworkError(f'Ошибка при загрузке файла: {response.status_code}')


    def load_dataset(
        self,
        dataset_url,
        local_file = None,
        local_path='datasets',
        sep=',',
        decimal='.'
    ):
        if local_file is None:
            local_file = dataset_url.split('/')[-1]
        local_dataset_file = f'{local_path}/{local_file}'
        remote_dataset_url = dataset_url
        def read_dataset_csv():
            return pd.read_csv(local_dataset_file, sep=sep, decimal=decimal)

        try:
            df = read_dataset_csv()
            print(f'Датасет успешно загружен из {local_dataset_file}')
        except FileNotFoundError:
            self.load_resource(remote_dataset_url, local_path, local_file)
            df = read_dataset_csv()

        print(f'Размер загруженного датасета: {df.shape[0]} строк, {df.shape[1]} столбцов', )
        return df

def df_info(df, name = '', n_samples=3):
    print('-'*50)
    print(f'Описание датасета {name}:')
    print('-'*50)
    print(df.info())
    print('-'*50)
    print(f'Данные датасета {name}:')
    print('-'*50)
    display(
        pd.concat([
            df.head(n_samples).assign(place='head'),
            df.sample(n_samples, random_state=RANDOM_STATE).assign(place='random'),
            df.tail(n_samples).assign(place='tail'),
        ]).sort_index()
    )

#### EDA
class CorrelationDisplayer:
    """
    Класс для отображения матрицы корреляций признаков в разных видах
    """
    def __init__(self, corr_matrix):
        self.corr_matrix = corr_matrix

    def get_corr_matrix(self):
        return self.corr_matrix

    def _subset_corr_matrix(self, subset):
        subset_cols = self.corr_matrix.columns if subset is None else subset
        return self.corr_matrix.loc[subset_cols, subset_cols]

    def draw_corr_matrix_full(
            self,
            digits=2,
            title='Матрица корреляций признаков',
            subtitle=None,
            figsize=(16, 10),
            subset=None,
    ):
        plt.subplots(figsize=figsize)
        matrix = self._subset_corr_matrix(subset)
        sns.heatmap(matrix.round(digits), annot=True, cmap='coolwarm', linewidths=0.5)
        plt.title(title + "\n" + subtitle if subtitle else title)
        plt.show()


    def draw_corr_matrix_with_target(
            self,
            target_col,
            title='Матрица корреляций с таргетом',
            subtitle=None,
            figsize=(16, 10),
    ):
        plt.subplots(figsize=figsize)
        data_heatmap = self.corr_matrix.loc[
            self.corr_matrix.index != target_col
        ][[target_col]].sort_values(by=target_col, ascending=False)
        sns.heatmap(data_heatmap, annot=True, cmap='coolwarm', linewidths=0.5)
        plt.title(title + "\n" + subtitle if subtitle else title)
        plt.show()

    def draw_pair_correlations(self, subset=None, figsize=(16, 10), corr_threshold=0.9):
        # преобразуем матрицу корреляции в датафрейм попарных корреляций
        # feature_1+feature_2 -> correlation
        matrix = self._subset_corr_matrix(subset)
        pair_correlations = matrix \
            .stack() \
            .reset_index() \
            .rename(columns={
                'level_0': 'feature1',
                'level_1': 'feature2',
                0: 'correlation'
            }) \
            .query('feature1 != feature2') \
            .sort_values(by='correlation', ascending=False) \

        def order_pair(row):
            if row['feature1'] > row['feature2']:
                return row['feature2'] + '/' + row['feature1']
            else:
                return row['feature1'] + '/' + row['feature2']

        pair_correlations['order_pair'] = pair_correlations.apply(order_pair, axis=1)
        pair_correlations = pair_correlations.drop(columns=['feature1', 'feature2'])
        pair_correlations = pair_correlations.drop_duplicates().reset_index(drop=True)
        pair_correlations = pair_correlations.query('correlation > @corr_threshold')
        pair_correlations = pair_correlations.sort_values(by='correlation')
        pair_correlations.plot(
            x='order_pair',
            y='correlation',
            xlabel='Значение корреляции',
            ylabel='Пара признаков',
            kind='barh',
            legend=False,
            figsize=figsize,
            grid=True,
        )
        plt.title('Попарные корреляции')
        plt.show()
        return pair_correlations.sort_values(by='correlation', ascending=False).reset_index(drop=True)

class EDAHelper:
    def __init__(self):
        pass

    def box_hist(self, df, column, title=None, bins=20, hue=None, kde=True, stat='density'):
        f, (ax_box, ax_hist) = plt.subplots(2, sharex=True, gridspec_kw={"height_ratios": (.15, .85)})
        print(df[[column]].describe())
        sns.boxplot(df[column], orient='h', ax=ax_box)
        sns.histplot(data=df, x=column, ax=ax_hist, bins=bins, hue=hue, kde=kde, stat=stat)

        f.suptitle(f'Распределение признака {column}' if title is None else title)
        ax_box.set(xlabel='')
        ax_hist.set(
            xlabel=f'Значения признака {column}',
            ylabel='Плотность распределения'
        )
        plt.show()

    def drop_duplicates(self, df, subset=None):
        ndups = df.duplicated(subset=subset).sum()
        if ndups > 0:
            df_orig = df.copy()
            df.drop_duplicates(subset=subset, inplace=True)
            diff = len(df_orig) - len(df)
            diff_pct = diff / len(df_orig) * 100
            print(f'Удалено {diff} строк ({diff_pct:.1f}%) из {len(df_orig)}')
        else:
            print('Дубликатов не обнаружено')

    def na_info(self, df, round_digits=1):
        '''
        Возвращает таблицу с количеством и процентом пропусков в столбцах датасета.
        '''
        count_na_name = 'Количество пропусков'
        res = pd.DataFrame({
            'Количество строк': len(df),
            count_na_name: df.isna().sum(),
            'Процент пропусков': round(df.isna().mean()*100, round_digits)
        }).sort_values(by=count_na_name, ascending=False)
        return res.query(f'`{count_na_name}` > 0').reset_index()


    # Уникальные значения всех категориальных признаков
    def print_unique_values(self, df, top_n=5):
        print('Уникальные значения всех категориальных признаков:\n')
        for col in df.select_dtypes(include=['object']).columns:
            unique_vals = df[col].unique().tolist()
            unique_vals.sort()
            top_n_vals = ', '.join(unique_vals[:top_n])
            unique_val_str =  top_n_vals if len(unique_vals) <= top_n else f'{top_n_vals}, ...'
            print(f'{col} [{df[col].nunique()}]: {unique_val_str}')

### Обучение модели
class ModelTrainHelper:
    def __init__(self):
        self.best_estimator_ = None
        pass

    def do_cross_validation(
            self,
            pipelines,
            X_train_val, y_train_val,
            scoring,
            metrics_df_list = [],
            return_train_score=False,
            cv=5,
            digits=3,
    ):

        cv_results_by_model = {}
        # Обучение моделей
        for name, p in pipelines.items():
            cv_results = cross_validate(
                estimator=p,
                X=X_train_val,
                y=y_train_val,
                scoring=scoring,
                return_train_score=return_train_score,
                return_estimator=True,
                cv=cv,
                verbose=0,
                n_jobs=-1,
            )
            cv_results_by_model[name] = cv_results

        def non_negative_metric(metric):
            if metric.startswith('neg_'):
                return (True, metric[len('_neg'):])
            else:
                return (False, metric)

        is_neg_main_metric, main_metric = non_negative_metric(scoring[0])

        # Сохранение результатов в сводную таблицу:
        def append_metrics(result_metrics, model_name, cv_results, test_or_train='test', scoring=[]):
            metrics_dict = {}

            metrics_dict['model_name'] = model_name
            metrics_dict = metrics_dict | {
                metric: np.mean(cv_results[f'{test_or_train}_{metric}']) for metric in scoring
            }
            # for metric in scoring:
            #     print(metric, cv_results[f'{test_or_train}_{metric}'])
            # add standard deviation
            # metrics_dict = metrics_dict | {
            #     f'{metric}_std': np.std(-cv_results[f'{test_or_train}_{metric}']) for metric in scoring
            # }

            keys = list(metrics_dict.keys())
            for metric in keys:
                # invert sign for neg metrics like neg_mean_squared_error
                # and rename metrics withoud neg
                if metric.startswith('neg_'):
                    metrics_dict[metric[len('neg_'):]] = metrics_dict[metric] * -1 if not metric.endswith('_std') else 1
                    metrics_dict.pop(metric)

            result_metrics.append(metrics_dict)
            return metrics_dict

        for model_name, model_cv_results in cv_results_by_model.items():
            append_metrics(
                metrics_df_list,
                model_name,
                model_cv_results,
                scoring=scoring,
            )
            if return_train_score and 'dummy' not in model_name.lower():
                append_metrics(
                    metrics_df_list,
                    f'{model_name} (train)',
                    model_cv_results,
                    test_or_train='train',
                    scoring=scoring,
                )

        metrics_df = pd.DataFrame(metrics_df_list) \
            .sort_values(by=main_metric, ascending=is_neg_main_metric)

        metrics_df.set_index('model_name', inplace=True)
        return metrics_df.sort_index(axis=1)

    def feature_importance(self, model, feature_names):
        # Получаем коэффициенты
        coefficients = model.coef_[0]
        intercept = model.intercept_[0]

        # DataFrame для анализа для удобства анализа коэффициентов
        coef_df = pd.DataFrame({
            'feature': feature_names,
            'coefficient': coefficients,
            'abs_coefficient': np.abs(coefficients)
        }).sort_values('abs_coefficient', ascending=False)

        # Визуализируем важность признаков:
        plt.figure(figsize=(8, 10))
        top_features = coef_df.sort_values(by='abs_coefficient', ascending=True)
        plt.barh(range(len(top_features)), top_features['coefficient'])
        plt.yticks(range(len(top_features)), top_features['feature'])
        plt.xlabel('Значение коэффициента')
        plt.title('Топ признаков по силе влияния на предсказание')
        plt.tight_layout()
        plt.show()

        return {
            'weights': coef_df.reset_index(drop=True),
            'intercept': intercept
        }

    def compare_metrics(self, baseline, enhanced, name, digits=3, pct_digits=0):
        diff = enhanced - baseline
        diff_pct = diff / baseline * 100
        print(f'Улучшение метрики {name}:',
            f'{baseline}->{enhanced:.{digits}f} ({diff_pct:.{pct_digits}f}%)')

class OptunaHelper:
    def __init__(self, X, y, cv):
        self.X = X
        self.y = y
        self.cv = cv

    def fit_study(self, params_func, estimator_func, scorer_func, n_trials=30,
                  visualize=False, show_progress_bar=False):

        def objective(trial):
            # описываем, какие гиперпараметры будем подбирать и в каких диапазонах.
            params = params_func(trial)

            # пайплайн с подготовкой данных и моделью
            # с подобранными на этой итерации гиперпараметрами
            pipeline = estimator_func(params)

            #  Задаём кросс-валидацию
            scores = cross_val_score(
                pipeline,
                self.X,
                self.y,
                cv=self.cv,
                scoring=scorer_func
            )

            # Среднее значение метрики на кросс-валидации
            mean_score = scores.mean()

            # Сообщаем результат Optuna
            trial.report(mean_score, step=0)

            # Если результат плохой — прерываем итерацию
            if trial.should_prune():
                raise optuna.TrialPruned()

            # Возвращаем среднее значение метрики на кросс-валидации
            return mean_score

        # Фиксируем сид через семплер
        sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)

        study = optuna.create_study(
            direction='maximize',
            sampler=sampler,
        )

        study.optimize(
            objective,
            n_trials=n_trials, # число итераций
            show_progress_bar=show_progress_bar,
        )

        # Прогресс метрики по попыткам
        if (visualize):
            fig1 = optuna.visualization.plot_optimization_history(study)
            # Важность гиперпараметров
            fig2 = optuna.visualization.plot_param_importances(study)

            display(fig1)
            display(fig2)
        return study


class GridSearchHelper:
    def __init__(self, X, y, scoring, cv):
        self.X = X
        self.y = y
        self.scoring = scoring
        self.main_metric = scoring[0]
        self.cv = cv
        self.last_results_top_ = None

    def fit_grid_(self, estimator, param_grid, model_name):
        print(f'Обучение модели {model_name} c перебором параметров...')
        grid = GridSearchCV(
            estimator=estimator,
            param_grid=param_grid,
            scoring=self.scoring,
            cv=self.cv,
            refit=self.main_metric,
            n_jobs=-1,
            verbose=0,
        )
        grid.fit(self.X, self.y)
        return grid

    def display_top_combinations_(self, grid_result, top_n=10):

        results_df = pd.DataFrame(grid_result.cv_results_) \
            .sort_values(by=f'mean_test_{self.main_metric}', ascending=False) \
            .reset_index(drop=True)

        print(f"\nТоп комбинаций по {self.main_metric}:")
        displayable_columns = [x for x in results_df.columns if x.startswith('mean_test_')]
        displayable_columns.insert(0, 'params')
        displayable_columns.append(f'std_test_{self.main_metric}')
        self.last_results_top_ = results_df[displayable_columns]
        display(self.last_results_top_.head(top_n))

    def fit_and_display_top(self, estimator, param_grid, model_name, top_n=5):
        grid_result = self.fit_grid_(estimator, param_grid, model_name)
        self.display_top_combinations_(grid_result, top_n=top_n)

        return grid_result

    def get_last_results_top(self):
        return self.last_results_top_

### Загрузка таблиц

In [5]:
loader = ResourceLoader()

df_loan_payment_credit = loader.load_dataset('https://code.s3.yandex.net/datasets/ds_15_loan_payment_credit.csv')
df_transactions = loader.load_dataset('https://code.s3.yandex.net/datasets/ds_15_transactions.csv')
df_client_description = loader.load_dataset('https://code.s3.yandex.net/datasets/ds_15_client_description.csv')
df_credit_description = loader.load_dataset('https://code.s3.yandex.net/datasets/ds_15_credit_description.csv')
df_mortgage_presence = loader.load_dataset('https://code.s3.yandex.net/datasets/ds_15_mortgage_presence.csv')
df_credit_rating = loader.load_dataset('https://code.s3.yandex.net/datasets/ds_15_credit_rating.csv')
df_macro_data = loader.load_dataset('https://code.s3.yandex.net/datasets/ds_15_macro_data.csv')
df_cohort_grid = loader.load_dataset('https://code.s3.yandex.net/datasets/ds_15_cohort_grid.csv')

Датасет успешно загружен из datasets/ds_15_loan_payment_credit.csv
Размер загруженного датасета: 5500 строк, 3 столбцов
Датасет успешно загружен из datasets/ds_15_transactions.csv
Размер загруженного датасета: 577494 строк, 10 столбцов
Датасет успешно загружен из datasets/ds_15_client_description.csv
Размер загруженного датасета: 13500 строк, 5 столбцов
Датасет успешно загружен из datasets/ds_15_credit_description.csv
Размер загруженного датасета: 13500 строк, 3 столбцов
Датасет успешно загружен из datasets/ds_15_mortgage_presence.csv
Размер загруженного датасета: 6609 строк, 3 столбцов
Датасет успешно загружен из datasets/ds_15_credit_rating.csv
Размер загруженного датасета: 577494 строк, 3 столбцов
Датасет успешно загружен из datasets/ds_15_macro_data.csv
Размер загруженного датасета: 84 строк, 4 столбцов
Датасет успешно загружен из datasets/ds_15_cohort_grid.csv
Размер загруженного датасета: 577494 строк, 2 столбцов


<a id="explore_data"></a>

## Исследовательский анализ данных

* Проведите первичный анализ данных:
    * Проверьте разные характеристики данных.
    * Исследуйте с помощью графиков количественные и категориальные данные.
    * Рекомендуем создать для этого функции, но это необязательное требование.

* Сделайте выводы о выбросах, пропусках, дубликатах и иных аномалиях в данных из каждой таблицы.

* Предобработка данных или их трансформация в этом проекте необязательны — кроме действий, необходимых для объединения таблиц.

<a id="dataset"></a>

## Формирование датасета

### Объединение таблиц

Соберите все источники данных о клиентах в единую таблицу наблюдений.

#### Формирование целевой переменной

1. Значение бинарной целевой переменной нужно определить для каждой строки со столбцами `ID` и `score_date` в таблице `cohort_grid`.

2. Таргет равен 1 при соблюдении двух условий:
    * Если значение в поле `просрочка_дней` больше или равно 90.
    * Если для клиента существует строка в таблице `loan_payment_credit`, где значение в поле `дата_начала_периода` попадает в интервал `[score_date, score_date + 365 дней)`.

>Важно: у клиента может быть несколько эпизодов с просрочками от 90 дней. Вам нужно взять первый по времени возникновения эпизод в таблице с просрочками.

3. После расчёта целевой переменной удалите строки, где дефолт уже произошёл к моменту скоринга, то есть `дата_начала_периода < score_date`. Это необходимо, так как для корректной работы с временной структурой важно учитывать дефолты, произошедшие в прошлом относительно даты скоринга.

#### Создание итоговой таблицы

1. В качестве признаков можно использовать только информацию о прошлом, то есть она должна быть доступна к дате скоринга. Иными словами, в каждой строке нужно присоединить данные о поведении клиента за предыдущие периоды относительно даты скоринга, иначе произойдёт утечка данных из будущего.

>К примеру, `score_date = 2024-01-15`. Тогда:
>* транзакции за декабрь 2023 г. — можем использовать;
>* транзакции за ноябрь 2023 г. — можем использовать;
>* транзакции 16 января 2024 г. — **не** можем использовать.

2. Присоедините остальные данные по клиенту, помимо указанных выше данных о макроэкономике и транзакциях клиента.

> Рекомендации:
>* Не забывайте проверять правильность каждого этапа сбора данных в единую таблицу. Это можно отслеживать на одном из клиентов.
>* При формировании таблицы следите за тем, чтобы в ней была корректно проведена работа со временем:
  >   * Отследите, не упущены ли какие-то данные из прошлого;
  >   * Проконтролируйте, верно ли рассчитана целевая переменная, которая зависит от дефолта в будущем.
>* Помните, что даты в исходных таблицах указаны на первое число месяца. Учитывайте период, который они описывают.

Сделайте выводы о получившейся таблице.

### Создание новых признаков

* Добавьте в таблицу новые признаки, которые помогли бы описать поведение клиента. Создайте не менее двух новых признаков.
* Сделайте выводы о новых признаках.

### Анализ итоговой таблицы

* Проведите краткий анализ получившейся итоговой таблицы.
* Сделайте вывод о данных для моделирования.
* Проверьте целевую переменную на предмет дисбаланса классов. Сделайте выводы.

<a id="modelling"></a>

## Моделирование

### Базовые модели

1. Подготовьте обучающую, калибровочную и тестовую выборки. Разбейте обучающую на три фолда для последующего использования кросс-валидации. Для оценки качества и калибровки используйте размер выборки, равный 12 месяцам.


2. При необходимости проведите категоризацию данных, применив нужный Encoder и использовав пайплайн.

3. Обучите базовые модели с кросс-валидацией по трём фолдам:
    * Две базовые модели — логистическую регрессию и случайный лес — без балансировки классов в целевой переменной.
    * Логистическую регрессию и случайный лес с балансировкой классов. Выберите метод балансировки самостоятельно. Обязательно примените хотя бы один метод. Можно попробовать несколько и выбрать лучший.
    * Сделайте выводы о работе всех четырёх моделей.

4. Случайный лес с настройками по умолчанию легко переобучается, потому что запоминает обучающую выборку, из-за чего модель может терять в качестве на новых данных. Логистическая регрессия же сразу готова к работе за счёт встроенной L2-регуляризации, которая автоматически контролирует сложность модели.

   Чтобы исправить проблемы модели Random Forest, вам нужно подобрать для неё гиперпараметры с помощью  Optuna. Количество гиперпараметров должно быть не менее трёх. Для оптимизации используйте метрику missed defaults rate.

5. Сравните все полученные модели.

6. Для оценки моделей используйте метрики:
   * accuracy или ROC-AUC,
   * approval rate,
   * default rate,
   * missed defaults rate.

7. Сделайте вывод о работе, проделанной в этом разделе.

<a id="calibration"></a>

## Калибровка модели и пересчёт результатов

* Проведите калибровку лучшей версии модели. Используйте отдельную калибровочную выборку.
* Используйте метод, подходящий для случайного леса.
* Постройте график калибровки.
* Сделайте вывод, оцените результаты с помощью коэффициента Бриера.

<a id="find_threshold"></a>

## Поиск порога решения

* Используя откалиброванную модель и калибровочную выборку, найдите порог, при котором будут достигнуты заданные в постановке задачи значения метрик:
    * approval rate — не менее 65%;
    * default rate — не более 2%;
    * missed defaults rate — не более 4%.
    
* Сделайте вывод о достигнутых в этом разделе результатах.

<a id="conf_matrix"></a>

## Анализ матрицы ошибок

* Оцените стабильность модели на тестовых данных. Для этого постройте:
    * матрицу ошибок на калибровочных данных;
    * матрицу классификации на тестовых данных.
* Сделайте вывод о моделях, рассчитав классические метрики машинного обучения и указанные в ТЗ бизнес-метрики.
* Сделайте вывод о стабильности модели.

<a id="final_model"></a>

## Фиксирование итоговой модели

- Опишите лучшую модель и найденный порог классификации.


<a id="feature_importance"></a>

## Анализ важности признаков

* Проведите анализ важности признаков найденной модели на полных тренировочных данных.
* Используйте `feature_importances_` для найденной модели.
* Сделайте вывод о силе влияния признаков на дефолт.

<a id="final_report"></a>
## Выводы по проекту

Сделайте выводы по проекту. Можете использовать такой план:

1. Цель и задачи исследования.

2. Подготовка данных и выборок.

3. Поиск и настройка модели.

4. Калибровка вероятностей.

5. Оптимизация бизнес-порога.

6. Анализ важности признаков.

7. Финальный пайплайн.

8. Основные выводы и рекомендации для бизнеса.

